<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
1. Drive 연결
</h3>

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
2. Ultralytics 설치
</h3>

In [3]:
%pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.6 MB/s eta 0:00:0000:01


<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
3. 경로 설정
</h3>

In [4]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/AI_Project/wardy/ml/src"
)

# 새로 만든 v2 데이터셋
DATASET_ZIP = DRIVE_ROOT / "data/hazard_objects_v2.zip"

# 기존 100 epoch 학습 결과
BEST_MODEL = (
    DRIVE_ROOT
    / "export/hazard_objects_v1_full_v1/weights/best.pt"
)

print("v2 dataset:", DATASET_ZIP.exists())
print("best.pt:", BEST_MODEL.exists())

v2 dataset: True
best.pt: True


<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
4. v2 데이터셋 압축 해제
</h3>

In [5]:
import shutil
from pathlib import Path

EXTRACT_ROOT = Path("/content")
DATASET_ROOT = EXTRACT_ROOT / "hazard_objects_v2"

if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)

shutil.unpack_archive(
    str(DATASET_ZIP),
    str(EXTRACT_ROOT)
)

print("dataset:", DATASET_ROOT)
print("존재:", DATASET_ROOT.exists())

dataset: /content/hazard_objects_v2
존재: True


<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
5. 데이터 개수 확인
</h3>

In [6]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

for split in ["train", "val", "test"]:
    image_dir = DATASET_ROOT / "images" / split

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTS
    ]

    print(f"{split}: {len(images)}장")

train: 5527장
val: 686장
test: 569장


<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
6. data.yaml 확인
</h3>

In [ ]:
DATA_YAML = DATASET_ROOT / "data.yaml"

print(DATA_YAML.read_text())

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
절대경로 설정
</h3>

In [ ]:
yaml_text = f"""
path: {DATASET_ROOT}

train: images/train
val: images/val
test: images/test

names:
  0: scissors
  1: knife
  2: cutter
  3: syringe
"""

DATA_YAML.write_text(
    yaml_text.strip(),
    encoding="utf-8"
)

print(DATA_YAML.read_text())

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
7. 기존 best.pt 불러오기
</h3>

In [ ]:
from ultralytics import YOLO

model = YOLO(str(BEST_MODEL))

print(model.names)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
8. 파인튜닝
</h3>

In [ ]:
SAVE_DIR = DRIVE_ROOT / "export"

results = model.train(
    data=str(DATA_YAML),

    epochs=40,
    imgsz=640,
    batch=16,
    device=0,

    lr0=0.001,

    # 40 epoch 끝까지 학습
    patience=0,

    # 회전 augmentation 추가
    degrees=30,

    project=str(SAVE_DIR),
    name="hazard_objects_v2_finetune_v3",

    plots=True,
    save=True,
)

# Ultralytics may increment the run directory when the requested name exists.
# Use the directory created by this training run for evaluation.
RUN_V3 = Path(model.trainer.save_dir)
BEST_V3 = RUN_V3 / "weights" / "best.pt"

print("Training directory:", RUN_V3)
print("BEST_V3:", BEST_V3)
print("best.pt exists:", BEST_V3.exists())


<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
9. 파인튜닝 Best Model 불러오기
</h3>

In [ ]:
from pathlib import Path
from ultralytics import YOLO

assert BEST_V3.exists(), f"best.pt not found: {BEST_V3}"
print("이번 학습의 best.pt:", BEST_V3)

model_v3 = YOLO(str(BEST_V3))
print(model_v3.names)


<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
10. 학습 결과 그래프 확인
</h3>

In [ ]:
from IPython.display import Image, display

display(
    Image(filename=str(RUN_V3 / "results.png"))
)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
11. Confusion Matrix 확인
</h3>

In [ ]:
display(
    Image(filename=str(RUN_V3 / "confusion_matrix.png"))
)

display(
    Image(filename=str(RUN_V3 / "confusion_matrix_normalized.png"))
)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
12. Precision-Recall Curve 확인
</h3>

In [ ]:
display(
    Image(filename=str(RUN_V3 / "BoxPR_curve.png"))
)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
13. F1-Confidence Curve 확인
</h3>

In [ ]:
display(
    Image(filename=str(RUN_V3 / "BoxF1_curve.png"))
)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
14. Precision / Recall Confidence Curve 확인
</h3>

In [ ]:
display(
    Image(filename=str(RUN_V3 / "BoxP_curve.png"))
)

display(
    Image(filename=str(RUN_V3 / "BoxR_curve.png"))
)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
15. Test Dataset 성능 평가
</h3>

In [ ]:
DATA_YAML = Path(
    "/content/hazard_objects_v2/data.yaml"
)

metrics_v3 = model_v3.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
    plots=True,
)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
16. Test mAP 결과 확인
</h3>

In [ ]:
print(f"mAP50    : {metrics_v3.box.map50:.4f}")
print(f"mAP75    : {metrics_v3.box.map75:.4f}")
print(f"mAP50-95 : {metrics_v3.box.map:.4f}")

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
17. 클래스별 mAP 확인
</h3>

In [ ]:
for class_id, class_map in enumerate(metrics_v3.box.maps):

    class_name = model_v3.names[class_id]

    print(
        f"{class_id} {class_name:10s}: "
        f"{class_map:.4f}"
    )

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
18. 전체 평가 결과 확인
</h3>

In [ ]:
print(metrics_v3.results_dict)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
19. Confusion Matrix 수치 확인
</h3>

In [ ]:
metrics_v3.confusion_matrix.to_df()

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
20. Test 이미지 랜덤 추론
</h3>

In [ ]:
import random
import matplotlib.pyplot as plt

TEST_DIR = DATASET_ROOT / "images/test"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

test_images = [
    p for p in TEST_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTS
]

print("Test 이미지:", len(test_images))

random.seed(42)

samples = random.sample(
    test_images,
    min(20, len(test_images))
)

for img_path in samples:

    results = model_v3.predict(
        source=str(img_path),
        imgsz=640,
        conf=0.4,
        iou=0.5,
        device=0,
        verbose=False,
    )

    annotated = results[0].plot()

    plt.figure(figsize=(10, 7))
    plt.imshow(annotated[..., ::-1])
    plt.title(img_path.name)
    plt.axis("off")
    plt.show()

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
21. 낮은 Confidence 탐지 결과 확인
</h3>

In [ ]:
test_image = random.choice(test_images)

results = model_v3.predict(
    source=str(test_image),
    imgsz=640,
    conf=0.01,
    device=0,
    verbose=False,
)

print("이미지:", test_image.name)

for box in results[0].boxes:

    cls_id = int(box.cls[0])
    confidence = float(box.conf[0])

    print(
        f"{model_v3.names[cls_id]:10s} "
        f"{confidence:.3f}"
    )

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
22. V1 / V2 모델 Test 평가
</h3>

In [ ]:
BEST_V1 = Path(
    "/content/drive/MyDrive/AI_Project/wardy/ml/src/export/"
    "hazard_objects_v1_full_v1/weights/best.pt"
)

BEST_V2 = Path(
    "/content/drive/MyDrive/AI_Project/wardy/ml/src/export/"
    "hazard_objects_v2_finetune_v2/weights/best.pt"
)

model_v1 = YOLO(str(BEST_V1))
model_v2 = YOLO(str(BEST_V2))

metrics_v1 = model_v1.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
    plots=False,
)

metrics_v2 = model_v2.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
    plots=False,
)

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
23. V1 / V2 / V3 전체 성능 비교
</h3>

In [ ]:
print("========== V1 ==========")
print(f"mAP50    : {metrics_v1.box.map50:.4f}")
print(f"mAP75    : {metrics_v1.box.map75:.4f}")
print(f"mAP50-95 : {metrics_v1.box.map:.4f}")

print()

print("========== V2 ==========")
print(f"mAP50    : {metrics_v2.box.map50:.4f}")
print(f"mAP75    : {metrics_v2.box.map75:.4f}")
print(f"mAP50-95 : {metrics_v2.box.map:.4f}")

print()

print("========== V3 ==========")
print(f"mAP50    : {metrics_v3.box.map50:.4f}")
print(f"mAP75    : {metrics_v3.box.map75:.4f}")
print(f"mAP50-95 : {metrics_v3.box.map:.4f}")

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
24. V1 / V2 / V3 클래스별 성능 비교
</h3>

In [ ]:
print(
    f"{'Class':10s} "
    f"{'V1':>8s} "
    f"{'V2':>8s} "
    f"{'V3':>8s} "
    f"{'V3-V1':>8s}"
)

for class_id in range(4):

    name = model_v3.names[class_id]

    v1_map = metrics_v1.box.maps[class_id]
    v2_map = metrics_v2.box.maps[class_id]
    v3_map = metrics_v3.box.maps[class_id]

    diff = v3_map - v1_map

    print(
        f"{name:10s} "
        f"{v1_map:8.4f} "
        f"{v2_map:8.4f} "
        f"{v3_map:8.4f} "
        f"{diff:+8.4f}"
    )

<h3 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;">
V1 / V2 / V3 동일 이미지 추론 비교
</h3>

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import random
import matplotlib.pyplot as plt

# ==========================================
# 모델
# ==========================================

print("V1:", BEST_V1.exists())
print("V2:", BEST_V2.exists())
print("V3:", BEST_V3.exists())


# ==========================================
# Test 이미지
# ==========================================

TEST_DIR = DATASET_ROOT / "images/test"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

test_images = [
    p for p in TEST_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTS
]

print("Test 이미지:", len(test_images))


# ==========================================
# 랜덤 이미지 선택
# ==========================================

NUM_SAMPLES = 20

random.seed(42)

samples = random.sample(
    test_images,
    min(NUM_SAMPLES, len(test_images))
)


# ==========================================
# V1 / V2 / V3 비교
# ==========================================

CONF = 0.4

for img_path in samples:

    result_v1 = model_v1.predict(
        source=str(img_path),
        imgsz=640,
        conf=CONF,
        iou=0.5,
        device=0,
        verbose=False,
    )[0]

    result_v2 = model_v2.predict(
        source=str(img_path),
        imgsz=640,
        conf=CONF,
        iou=0.5,
        device=0,
        verbose=False,
    )[0]

    result_v3 = model_v3.predict(
        source=str(img_path),
        imgsz=640,
        conf=CONF,
        iou=0.5,
        device=0,
        verbose=False,
    )[0]

    img_v1 = result_v1.plot()[..., ::-1]
    img_v2 = result_v2.plot()[..., ::-1]
    img_v3 = result_v3.plot()[..., ::-1]

    fig, axes = plt.subplots(1, 3, figsize=(24, 7))

    axes[0].imshow(img_v1)
    axes[0].set_title("V1")
    axes[0].axis("off")

    axes[1].imshow(img_v2)
    axes[1].set_title("V2")
    axes[1].axis("off")

    axes[2].imshow(img_v3)
    axes[2].set_title("V3")
    axes[2].axis("off")

    fig.suptitle(
        f"{img_path.name}   |   conf={CONF}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()
